## Train the Neural Network (MLP) model

### 1. Hyperparameter tuning

Each Optuna trial picks a value for every hyperparameter, trains the model on the 3 validation folds,
and returns the mean Poisson deviance (lower is better). <br> Optuna uses the results to pick better values in the next trials.

The test set is not used here.

In [1]:
# 1. get the repo + the PR branch (the data is already in it)
!git clone -q https://github.com/amelia-lee-smu/CS610-Applied-Machine-Learning.git
%cd CS610-Applied-Machine-Learning
!git fetch -q origin pull/6/head:pr6 && git checkout -q pr6

# 2. add the neural network model file and the seed constant
!cp /content/neural_network.py utilities/
!printf '\n# fixed seed for the models themselves\nRANDOM_SEED = 42\n' >> utilities/values.py

# 3. packages (Colab already has torch, pandas, scikit-learn, pyarrow)
!pip install -q optuna

# 4. the notebook expects to run from the notebooks/ folder
import os
os.chdir('/content/CS610-Applied-Machine-Learning/notebooks')

/content/CS610-Applied-Machine-Learning


In [2]:
import sys
from pathlib import Path

import optuna

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utilities.neural_network import *
from utilities.training import *
from utilities.utilities import *
from utilities.values import *

data_dir = '../data/'
model_dir = '../models/'

# read the train-validation set (Jan 2020 - Feb 2021)
train_validation_df = read_parquet_file(data_dir + 'processed/train_validation.parquet')

In [3]:
def objective(trial):
    # 1. let optuna pick the hyperparams for this trial
    hyperparams = {
        'station_dim': trial.suggest_categorical('station_dim', [4, 8, 16]),
        'hidden_size': trial.suggest_categorical('hidden_size', [32, 64, 128]),
        'n_layers': trial.suggest_int('n_layers', 1, 3),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-2, log = True),     # search on a log scale
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log = True),
        'epochs': trial.suggest_int('epochs', 8, 20),
        'batch_size': trial.suggest_categorical('batch_size', [1024, 2048, 4096]),
    }

    # 2. train a new model with these hyperparams on each validation fold, and score it
    scores_df = cross_validate(NeuralNetworkModel, hyperparams, train_validation_df)
    mean_scores = scores_df[['MAE', 'RMSE', 'poisson_deviance']].mean()

    # Optuna tunes on the Poisson deviance only
    # MAE and RMSE are stored with the trial so they also appear in the results table
    trial.set_user_attr('MAE', mean_scores['MAE'])
    trial.set_user_attr('RMSE', mean_scores['RMSE'])

    # show the scores of this trial
    print('Poisson deviance per fold:', scores_df['poisson_deviance'].round(4).tolist())
    print(f'Trial {trial.number}: mean Poisson deviance = {mean_scores["poisson_deviance"]:.4f}, '
          f'mean MAE = {mean_scores["MAE"]:.4f}, mean RMSE = {mean_scores["RMSE"]:.4f}\n')
    return mean_scores['poisson_deviance']


# hide Optuna's own log line for every trial (the progress bar below shows the progress instead)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# the seed makes the search repeatable
study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler(seed = OPTUNA_SEED))

In [4]:
n_trials = 20

# the progress bar shows how many trials are done, the time remaining and the best score so far
study.optimize(objective, n_trials = n_trials, show_progress_bar = True)

print('Best hyperparameters:', study.best_params)
print('Best mean Poisson deviance:', round(study.best_value, 4))
print('Mean MAE of the best trial:', round(study.best_trial.user_attrs['MAE'], 4))
print('Mean RMSE of the best trial:', round(study.best_trial.user_attrs['RMSE'], 4))

  0%|          | 0/20 [00:00<?, ?it/s]

Fold 1: training NeuralNetworkModel on 298605 rows
Fold 2: training NeuralNetworkModel on 373269 rows
Fold 3: training NeuralNetworkModel on 447933 rows
Poisson deviance per fold: [1.1018, 0.8885, 0.7585]
Trial 0: mean Poisson deviance = 0.9163, mean MAE = 0.5268, mean RMSE = 0.9244

Fold 1: training NeuralNetworkModel on 298605 rows
Fold 2: training NeuralNetworkModel on 373269 rows
Fold 3: training NeuralNetworkModel on 447933 rows
Poisson deviance per fold: [1.1064, 0.8699, 0.7899]
Trial 1: mean Poisson deviance = 0.9221, mean MAE = 0.5323, mean RMSE = 0.9784

Fold 1: training NeuralNetworkModel on 298605 rows
Fold 2: training NeuralNetworkModel on 373269 rows
Fold 3: training NeuralNetworkModel on 447933 rows
Poisson deviance per fold: [1.0965, 0.8902, 0.6579]
Trial 2: mean Poisson deviance = 0.8815, mean MAE = 0.5383, mean RMSE = 0.9381

Fold 1: training NeuralNetworkModel on 298605 rows
Fold 2: training NeuralNetworkModel on 373269 rows
Fold 3: training NeuralNetworkModel on 4479

In [5]:
# all trials, best first
trials_df = study.trials_dataframe()
trials_df.sort_values('value').head(10)

,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_dropout,params_epochs,params_hidden_size,params_learning_rate,params_n_layers,params_station_dim,params_weight_decay,user_attrs_MAE,user_attrs_RMSE,system_attrs_tpe:relative_params:0,state
11,11,0.867378,2026-10-30 04:59:02.304128,2026-10-30 05:00:45.673919,0 days 00:01:43.369791,1024,0.011121,20,32,0.002867,3,4,0.000002,0.531774,0.967299,"{""batch_size"": 1024, ""dropout"": 0.011121331424...",COMPLETE
5,5,0.872180,2026-10-30 04:53:02.531116,2026-10-30 04:54:47.462877,0 days 00:01:44.931761,1024,0.022213,19,32,0.002283,3,4,0.000002,0.533953,0.971089,NaN,COMPLETE
14,14,0.878507,2026-10-30 05:03:48.375460,2026-10-30 05:05:06.479308,0 days 00:01:18.103848,1024,0.008740,19,32,0.002019,2,4,0.000010,0.536084,0.974844,"{""batch_size"": 1024, ""dropout"": 0.008739732846...",COMPLETE
13,13,0.880910,2026-10-30 05:02:11.020520,2026-10-30 05:03:48.374061,0 days 00:01:37.353541,1024,0.014675,19,32,0.003229,3,4,0.000001,0.539205,0.978474,"{""batch_size"": 1024, ""dropout"": 0.014675023790...",COMPLETE
2,2,0.881548,2026-10-30 04:46:56.465873,2026-10-30 04:50:59.418256,0 days 00:04:02.952383,4096,0.242519,16,128,0.002017,3,16,0.000002,0.538262,0.938140,NaN,COMPLETE
17,17,0.885030,2026-10-30 05:11:24.884544,2026-10-30 05:12:32.558853,0 days 00:01:07.674309,1024,0.023603,13,32,0.003720,3,4,0.000003,0.550866,0.985839,"{""batch_size"": 1024, ""dropout"": 0.023603095759...",COMPLETE
19,19,0.887149,2026-10-30 05:13:53.375742,2026-10-30 05:15:15.469966,0 days 00:01:22.094224,1024,0.017466,16,32,0.006080,3,4,0.000004,0.531771,1.015561,"{""batch_size"": 1024, ""dropout"": 0.017465820975...",COMPLETE
16,16,0.887437,2026-10-30 05:06:30.923761,2026-10-30 05:11:24.882979,0 days 00:04:53.959218,1024,0.077531,19,128,0.005343,3,4,0.000004,0.540126,1.010321,"{""batch_size"": 1024, ""dropout"": 0.077531382597...",COMPLETE
7,7,0.890933,2026-10-30 04:55:53.599175,2026-10-30 04:56:37.231828,0 days 00:00:43.632653,2048,0.074788,10,32,0.002573,3,8,0.000185,0.529285,0.954838,NaN,COMPLETE
15,15,0.896519,2026-10-30 05:05:06.480801,2026-10-30 05:06:30.922241,0 days 00:01:24.441440,1024,0.071013,16,32,0.001538,3,4,0.000003,0.553777,0.949691,"{""batch_size"": 1024, ""dropout"": 0.071012625172...",COMPLETE


### 2. Validation predictions (for the ensemble)

- Get the best set of hyperparameters from optuna.
- For each of the cross-validation sets, train the model on the train set, and store the validation set predictions.
- These validation set predictions will be used to train the ensemble model later.

In [6]:
# get the predictions of each validation fold
fold_results = get_out_of_fold_predictions(NeuralNetworkModel, study.best_params, train_validation_df)

# save the predictions in each file separately
for fold_number, fold_df in enumerate(fold_results, start = 1):
    save_to_parquet_file(fold_df, data_dir + f'predictions/validation_neural_network_fold{fold_number}.parquet')
print('Validation predictions saved.')

Fold 1: training NeuralNetworkModel on 298605 rows


/content/CS610-Applied-Machine-Learning/utilities/training.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df[PREDICTION] = model.predict(validation_df)
/content/CS610-Applied-Machine-Learning/utilities/training.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['fold'] = fold_number


Fold 2: training NeuralNetworkModel on 373269 rows


/content/CS610-Applied-Machine-Learning/utilities/training.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df[PREDICTION] = model.predict(validation_df)
/content/CS610-Applied-Machine-Learning/utilities/training.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['fold'] = fold_number


Fold 3: training NeuralNetworkModel on 447933 rows


/content/CS610-Applied-Machine-Learning/utilities/training.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df[PREDICTION] = model.predict(validation_df)
/content/CS610-Applied-Machine-Learning/utilities/training.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['fold'] = fold_number


Validation predictions saved.


### 3. Final model and test set prediction

In [7]:
test_df = read_parquet_file(data_dir + 'processed/test.parquet')

In [8]:
# train a new model on the entire train set (Jan 2020 - Feb 2021) using the best hyperparams from Optuna
model = NeuralNetworkModel(**study.best_params)
model.fit(train_validation_df)

# do predictions on the test set (Mar -April 2021)
test_df[PREDICTION] = model.predict(test_df)

# save the model and results
model.save_model(model_dir + 'neural_network_model.pkl')
save_to_parquet_file(test_df, data_dir + 'predictions/test_neural_network.parquet')
print('Model and test predictions saved.')

Model and test predictions saved.


In [9]:
# test set scores (for reporting only: the test set was not used to choose any hyperparameter)
test_scores = evaluate(model, test_df)
print({name: round(score, 4) for name, score in test_scores.items()})

{'MAE': 0.5106, 'RMSE': 0.9718, 'poisson_deviance': 0.9729}


In [10]:
test_df.head()

,date,num_rentals,start station name,hour_of_day,day_of_week,num_rentals_t-2,num_rentals_t-3,sine_week_of_year,cosine_week_of_year,temperature_2m,snow_depth,cloud_cover,cloud_cover_mid,relative_humidity_2m,precipitation,wind_speed_10m,is_public_holiday,prediction
0,2021-03-01,0,5 Corners Library,0,0,0.0,0.0,0.812487,0.582979,5.55,0.1,100.0,99.0,96.586780,0.5,10.787993,0,0.015834
1,2021-03-01,0,5 Corners Library,1,0,0.0,0.0,0.812487,0.582979,5.40,0.1,100.0,100.0,97.596886,0.7,12.181624,0,0.005655
2,2021-03-01,0,5 Corners Library,2,0,0.0,0.0,0.812487,0.582979,5.45,0.1,100.0,100.0,97.597850,0.4,9.164103,0,0.003237
3,2021-03-01,0,5 Corners Library,3,0,0.0,0.0,0.812487,0.582979,5.45,0.1,100.0,100.0,97.597850,0.5,6.952755,0,0.002254
4,2021-03-01,0,5 Corners Library,4,0,0.0,0.0,0.812487,0.582979,5.05,0.1,100.0,100.0,98.273410,1.0,8.121970,0,0.002477


In [11]:
!cd .. && zip -r neural_network_results.zip models/neural_network_model.pkl data/predictions/*neural_network*.parquet
from google.colab import files
files.download('../neural_network_results.zip')

  adding: models/neural_network_model.pkl (deflated 29%)
  adding: data/predictions/test_neural_network.parquet (deflated 39%)
  adding: data/predictions/validation_neural_network_fold1.parquet (deflated 23%)
  adding: data/predictions/validation_neural_network_fold2.parquet (deflated 27%)
  adding: data/predictions/validation_neural_network_fold3.parquet (deflated 26%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>